# 11 - Order Book Imbalance

Bid vs ask depth.


## Setup

In [1]:
# --- setup --------------------------------------------------------
import sys, json
sys.path.insert(0, r"D:\\Projects\\done\\eth_bot\\eth_bot_v2")
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (13, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
np.random.seed(42)

SYMBOL = "ETHUSDT"
OUT = Path(r"D:\\Projects\\done\\eth_bot\\eth_bot_v2") / "reports" / "forecasts"
OUT.mkdir(parents=True, exist_ok=True)
DB = Path(r"D:\\Projects\\done\\eth_bot\\eth_bot_v2") / "database" / "market.db"

def sql(q, **p):
    with sqlite3.connect(DB) as c:
        return pd.read_sql_query(q, c, params=p)

from database.config.api.binance import get_klines

def fetch(symbol, interval, limit=500):
    raw = get_klines(symbol, interval=interval, limit=limit)
    df = pd.DataFrame(raw, columns=["open_time","open","high","low","close",
        "volume","close_time","quote_volume","trades","taker_base","taker_quote","ignore"])
    df["date"] = pd.to_datetime(df["open_time"], unit="ms", utc=True).dt.tz_convert(None)
    for c in ("open","high","low","close","volume"):
        df[c] = df[c].astype(float)
    return df[["date","open","high","low","close","volume"]].set_index("date").iloc[:-1]

def action_block(name, action, reason, price=None, entry=None, stop=None,
                 target=None, conf=None, extra=None):
    print("=" * 62)
    print(f"  {name}")
    print("=" * 62)
    print(f"  time   : {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M')} UTC")
    if price is not None: print(f"  price  : {price:,.2f}")
    if conf  is not None: print(f"  conf   : {conf:.2f}")
    print(f"  ACTION : {action}")
    print(f"  REASON : {reason}")
    if entry is not None:
        print(f"  entry  : {entry:,.2f}   stop: {stop:,.2f}   target: {target:,.2f}")
    if extra:
        for k, v in extra.items():
            print(f"  {k:<6} : {v}")
    print("=" * 62)
    return {"action": action, "reason": reason, "price": price,
            "entry": entry, "stop": stop, "target": target, "conf": conf}

## 1. Load orderbook snapshot

In [2]:
ob = sql("SELECT ts, bid_price, bid_qty, ask_price, ask_qty FROM orderbook ORDER BY ts DESC LIMIT 40")
print("rows:", len(ob))
if not ob.empty:
    latest_ts = ob.ts.max()
    snap = ob[ob.ts == latest_ts]
    bid_vol = snap.bid_qty.sum()
    ask_vol = snap.ask_qty.sum()
    imb = (bid_vol - ask_vol) / (bid_vol + ask_vol)
    print(f"bid vol: {bid_vol:.3f}")
    print(f"ask vol: {ask_vol:.3f}")
    print(f"imbalance: {imb:+.4f}")

rows: 20
bid vol: 37.575
ask vol: 0.000
imbalance: +1.0000


## 2. Action

In [3]:
if len(ob) == 0:
    action, reason = "FLAT", "no orderbook data"
elif imb > 0.3:   action, reason = "LONG",  "bids dominate — buy pressure"
elif imb < -0.3:  action, reason = "SHORT", "asks dominate — sell pressure"
else:             action, reason = "FLAT",  "balanced book"
action_block("ORDERBOOK", action, reason, extra={"imbalance": f"{imb:+.3f}" if len(ob) else "n/a"})

  ORDERBOOK
  time   : 2026-10-04 10:47 UTC
  ACTION : LONG
  REASON : bids dominate — buy pressure
  imbalance : +1.000


{'action': 'LONG',
 'reason': 'bids dominate — buy pressure',
 'price': None,
 'entry': None,
 'stop': None,
 'target': None,
 'conf': None}